In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ─── Analogs and layout ─────────────────────────────────────
ANALOGS  = ['scl', 'scc', 'scs', 'scy', 'scr', 'scd']
LABELS   = {'scl': 'LEU', 'scc': 'CYS', 'scs': 'SER',
            'scy': 'TYR', 'scr': 'ARG$^+$', 'scd': 'ASP$^-$'}
BLOCK_NS = [50, 100, 150, 200, 300, 600]

# ─── Data locations ─────────────────────────────────────────
RAW_DATA_DIR   = "../data/distribution_data/raw_data"  # last 600 ns (100 fps)
DIST_TOTAL_DIR = "../data/distribution_data/total"     # summary_{analog}.dat (null-density plateau)
TRAJS          = [1, 2, 3]
FRAMES_PER_NS  = 100

# ─── PMF constants ──────────────────────────────────────────
k_B      = 0.008314               # kJ/(mol·K)
T_PMF    = 303.15                 # K
EPSILON  = 1e-5
Z_BIN    = 1.0
Z_MAX    = 50.0
REF_LO, REF_HI = 40.0, 50.0        # bulk-water reference window
bin_edges   = np.arange(0.0, Z_MAX + Z_BIN, Z_BIN)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
ref_mask    = (bin_centers >= REF_LO) & (bin_centers <= REF_HI)

REGION_BOUNDS = [9.5, 19.5, 30.0]

# ─── Helpers ────────────────────────────────────────────────
def plateau_end(analog, z_max=20.0):
    """Return the upper z bound of the null-density plateau starting at z=0,
    looking up to z_max. Returns 0 if there is no plateau."""
    path = os.path.join(DIST_TOTAL_DIR, analog, f"summary_{analog}.dat")
    if not os.path.isfile(path):
        return 0.0
    df = pd.read_csv(path, sep=r'\s+')
    df = df[(df['z'] >= 0) & (df['z'] <= z_max)].sort_values('z').reset_index(drop=True)
    end = 0.0
    for _, row in df.iterrows():
        if row['mean'] == 0:
            end = row['z'] + 0.5
        else:
            break
    return end

def plot_pmf_with_plateau(ax, x, y, pe, color, lw, label=None, alpha=1.0):
    """Plot y vs x with a dashed segment on the null-density plateau (x ≤ pe)
    and a solid segment beyond it (x ≥ pe), sharing the same color. The
    legend label is attached only to the solid segment."""
    x = np.asarray(x)
    y = np.asarray(y)
    if pe > 0:
        mask_dash  = x <= pe
        mask_solid = x >= pe   # overlap at pe keeps the curve continuous
    else:
        mask_dash  = np.zeros_like(x, dtype=bool)
        mask_solid = np.ones_like(x, dtype=bool)
    if mask_solid.any():
        ax.plot(x[mask_solid], y[mask_solid], '-',
                color=color, lw=lw, alpha=alpha, label=label)
    if mask_dash.any():
        ax.plot(x[mask_dash], y[mask_dash], '--',
                color=color, lw=lw, alpha=alpha)

def _load_contacts(path):
    """Return (frame, z) arrays from a *_contacts_*.dat file."""
    if not os.path.isfile(path):
        return None, None
    arr = np.loadtxt(path, skiprows=1)
    return arr[:, 0].astype(np.int64), arr[:, 4]

def _load_raw_trajectory(analog, traj):
    """Load one trajectory from raw_data (last 600 ns, 100 fps)."""
    path = os.path.join(RAW_DATA_DIR, analog, f"{analog}_contacts_{traj}.dat")
    return _load_contacts(path)

def _pmf_from_z(z_abs, edges):
    counts, _ = np.histogram(z_abs, bins=edges)
    prob = counts / counts.sum()
    return -k_B * T_PMF * np.log(prob + EPSILON)

def _block_bounds(max_frame, block_frames):
    total = max_frame + 1
    n_blk = total // block_frames
    leftover = total - n_blk * block_frames
    if leftover >= 0.5 * block_frames:
        n_blk += 1
    n_blk = max(n_blk, 1)
    return [(b * block_frames, min((b + 1) * block_frames, total))
            for b in range(n_blk)]

def block_pmfs(frames, z_vals, block_frames, edges, ref_msk):
    """Reference offset from the full (unfiltered) data per block."""
    order = np.argsort(frames)
    fr_s  = frames[order]
    z_s   = np.abs(z_vals[order])
    bounds = _block_bounds(int(fr_s.max()), block_frames)
    out = np.full((len(bounds), len(edges) - 1), np.nan)
    for b, (lo, hi) in enumerate(bounds):
        i0 = np.searchsorted(fr_s, lo, side='left')
        i1 = np.searchsorted(fr_s, hi, side='left')
        z_block = z_s[i0:i1]
        if len(z_block) < 10:
            continue
        pmf_raw = _pmf_from_z(z_block, edges)
        ref     = pmf_raw[ref_msk].mean() if ref_msk.any() else 0.0
        out[b]  = pmf_raw - ref
    return out

# ─── Plot ───────────────────────────────────────────────────
n_cols, n_rows = 2, 3
fig, axes = plt.subplots(n_rows, n_cols,
                          figsize=(12, 2.2 * n_rows),
                          sharex=True, dpi=2000,
                          gridspec_kw={'hspace': 0.35, 'wspace': 0.20})
axes_flat = axes.flatten()

# viridis palette for the 5 shorter blocks; 600 ns gets orange
# (replaces the hard-to-see yellow tail of viridis).
_vir = plt.cm.viridis
BLOCK_COLORS = [_vir(v) for v in np.linspace(0.0, 0.75, len(BLOCK_NS)-1)] + ['orange']

for idx, analog in enumerate(ANALOGS):
    ax = axes_flat[idx]
    label = LABELS[analog]

    # Gather frames/z from the 3 trajectories (raw_data only).
    fr_all, z_all = [], []
    for t in TRAJS:
        fr, zv = _load_raw_trajectory(analog, t)
        if fr is None:
            continue
        fr_all.append(fr); z_all.append(zv)
    if not fr_all:
        ax.text(0.5, 0.5, 'no data', transform=ax.transAxes,
                ha='center', va='center', color='gray')
        ax.set_title(label, fontsize=10, fontweight='bold')
        continue

    pe = plateau_end(analog)
    plotted_series = []  # (color, mean_p, se_p) for zoom-inset replot
    for bs_ns in BLOCK_NS:
        bs_frames = int(bs_ns * FRAMES_PER_NS)
        stacked = []
        for fr, zv in zip(fr_all, z_all):
            arr = block_pmfs(fr, zv, bs_frames, bin_edges, ref_mask)
            valid = ~np.isnan(arr).any(axis=1)
            if valid.any():
                stacked.append(arr[valid])
        if not stacked:
            continue
        stacked = np.vstack(stacked)
        mean_p  = stacked.mean(axis=0)
        se_p    = stacked.std(axis=0, ddof=1) / np.sqrt(stacked.shape[0])
        color   = BLOCK_COLORS[BLOCK_NS.index(bs_ns)]
        plot_pmf_with_plateau(ax, bin_centers, mean_p, pe,
                              color=color, lw=1.5,
                              label=f'{bs_ns} ns (n={stacked.shape[0]})')
        ax.fill_between(bin_centers,
                        mean_p - 1.96 * se_p, mean_p + 1.96 * se_p,
                        color=color, alpha=0.15, linewidth=0)
        plotted_series.append((color, mean_p, se_p))

    for xv in REGION_BOUNDS:
        ax.axvline(x=xv, linestyle='--', alpha=0.7, linewidth=0.8,
                   color='gray', zorder=0)
    ax.set_xlim(0, 35)
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.grid(True, which='major', linestyle='--', alpha=0.4, linewidth=0.3)
    ax.tick_params(axis='both', labelsize=8)
    ax.legend(fontsize=7, ncol=2, frameon=False)

    # ── Zoom insets ──────────────────────────────────────────
    #   LEU : zoom on z ∈ [0.5, 3.5] Å, PMF ∈ [-25, -14] kJ/mol
    #         indicator box on main plot: z ∈ [0.5, 3.5],  y ∈ [-25, -14]
    #         zoom graph  on main plot:   z ∈ [0.5, 6.5],  y ∈ [-8, 2]
    #   TYR : zoom on z ∈ [10.5, 13.5] Å, PMF ∈ [-20, -13] kJ/mol
    #         indicator box on main plot: z ∈ [10.5, 13.5], y ∈ [-21, -13]
    #         zoom graph  on main plot:   z ∈ [8.5, 14.5],  y ∈ [-8, 2]
    #
    # 'inset_bounds' and 'box' are in DATA coordinates on the main plot
    # (x0, y0, width, height).
    ZOOM_SPECS = {
        'scl': {'xlim': (0.5, 3.5),   'ylim': (-25.0, -14.0),
                'box':          (0.5, -25.0, 3.0, 11.0),
                'inset_bounds': (3.5,  -10.0, 4.5, 16.5)},
        'scy': {'xlim': (10.5, 13.5), 'ylim': (-21.0, -13.0),
                'box':          (10.5, -21.0, 3.0, 8.0),
                'inset_bounds': ( 9.5,  -9.8, 4.5, 12.0)},
    }
    if analog in ZOOM_SPECS and plotted_series:
        spec = ZOOM_SPECS[analog]
        zx0, zx1 = spec['xlim']
        zy0, zy1 = spec['ylim']
        axins = ax.inset_axes(spec['inset_bounds'],
                              transform=ax.transData)
        for c, mp, sp in plotted_series:
            plot_pmf_with_plateau(axins, bin_centers, mp, pe,
                                  color=c, lw=1.2)
            axins.fill_between(bin_centers,
                               mp - 1.96 * sp, mp + 1.96 * sp,
                               color=c, alpha=0.15, linewidth=0)
        axins.set_xlim(zx0, zx1)
        axins.set_ylim(zy0, zy1)
        # Integer-only ticks on the zoom x-axis.
        axins.xaxis.set_major_locator(ticker.MultipleLocator(1))
        axins.xaxis.set_major_formatter(ticker.FormatStrFormatter('%d'))
        axins.yaxis.set_major_formatter(ticker.FormatStrFormatter('%d'))
        axins.tick_params(axis='both', labelsize=5)
        axins.grid(True, linestyle='--', alpha=0.3, linewidth=0.3)
        # Indicator rectangle marks the actual zoom window; connector
        # lines link it to the inset panel placed just above it.
        ax.indicate_inset(spec['box'], inset_ax=axins,
                          edgecolor='black', lw=0.8, alpha=0.8,
                          facecolor='none')

fig.text(0.5, 0.02, r"z (Å)", ha='center', fontsize=11)
fig.text(0.06, 0.5, "PMF (kJ/mol)", va='center', rotation='vertical', fontsize=11)

os.makedirs("../plot", exist_ok=True)
out = "../plot/FigureS9.png"
plt.tight_layout(rect=[0.04, 0.03, 1, 1])
plt.savefig(out, dpi=600, bbox_inches='tight')
plt.show()
print("Saved →", out)